# Hybrid Search

In Retrieval-Augmented Generation (RAG), retrieving the most relevant documents
is critical for generating accurate answers.

Hybrid search improves retrieval by combining:
- Dense (vector-based) search for semantic understanding
- Sparse (BM25) search for keyword matching

This notebook demonstrates how hybrid search performs re-ranking
by combining scores from both methods.

## What this notebook contains

1. Definition of documents and query
2. Vector-based search using a bi-encoder
3. Keyword-based search using BM25
4. Hybrid score-based re-ranking
5. Final ranked results
6. Observations


In [ ]:
pip install sentence-transformers rank-bm25 scikit-learn numpy


## Document Corpus

This cell defines a small set of documents that act as our knowledge base.
These documents simulate real-world information such as FAQs or policies.


In [ ]:
docs = [
    "AI course fee is 20,000 rupees for beginners",
    "Refund is available within 7 days",
    "Classes run from 9am to 4pm",
    "AI Course",
    "Advanced AI course costs 40,000 rupees"
]


In [ ]:
print("\nDOCUMENTS:")
for i, d in enumerate(docs):
    print(i, d)


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
doc_vecs = model.encode(docs)



## User Query

This cell defines the user query for which relevant documents
need to be retrieved and ranked.


In [ ]:
query = "Money Back Policy"
q_vec = model.encode([query])[0]


## Vector Search using Bi-Encoder

In this step:
- The query and documents are encoded separately
- Cosine similarity is used to measure semantic similarity
- This represents dense retrieval used in vector databases

This method is fast but provides approximate relevance.


In [ ]:
vec_scores = cosine_similarity([q_vec], doc_vecs)[0]


In [ ]:
print("\nVECTOR SEARCH SCORES:")
for i, score in enumerate(vec_scores):
    print(i, ":", docs[i], "=>", round(score, 3))


## Observation: Vector Search Results

- Documents with similar meaning to the query receive higher scores
- Exact keyword matching is not required
- Some partially relevant documents may appear higher in ranking


## Keyword Search using BM25

In this step:
- Documents are tokenized into words
- BM25 ranks documents based on keyword overlap
- This represents traditional information retrieval

BM25 is precise for keywords but weak for semantic meaning.


In [ ]:
from rank_bm25 import BM25Okapi


In [ ]:
tokenized = [d.lower().split() for d in docs]


In [ ]:
bm25 = BM25Okapi(tokenized)
bm25_scores = bm25.get_scores(query.lower().split())


In [ ]:
print("\nBM25 SCORES:")
for i, score in enumerate(bm25_scores):
    print(i, ":", docs[i], "=>", round(score, 3))


## Observation: BM25 Results

- Documents containing exact query terms score higher
- Documents without keyword overlap score very low
- Semantic similarity is not captured


## Hybrid Search

In this step:
- Scores from vector search and BM25 are combined
- Documents are re-ranked based on the combined score

This approach balances semantic understanding and keyword matching.


In [ ]:
hybrid_scores = vec_scores + bm25_scores


In [ ]:
best = np.argsort(hybrid_scores)[::-1]


## Final Ranked Documents

After hybrid re-ranking:
- The most relevant documents appear at the top
- Irrelevant documents move down the list
- The final ranking is more accurate than using a single method


In [ ]:
for i in best:
    print(docs[i])


## Final Observations

- Vector search captures meaning but may miss exact intent
- BM25 captures intent but misses semantics
- Hybrid search combines the strengths of both
- Re-ranking improves document relevance in RAG systems
